In [90]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import cv2

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input,Conv2D,MaxPool2D,Flatten,Dense,Dropout,concatenate
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.layers import BatchNormalization

from PIL import Image
import matplotlib.pyplot as plt



In [91]:
import zipfile

zip_path = "EuroSAT.zip"
extract_path = "EuroSAT"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Dataset extracted successfully")

Dataset extracted successfully


In [92]:
IMG_SIZE = 64

data = []
labels = []

classes = ["Industrial", "Residential", "Highway"]

dataset_path = "EuroSAT/2750"

for idx, cls in enumerate(classes):

    path = os.path.join(dataset_path, cls)

    for img in os.listdir(path)[:1000]:

        img_path = os.path.join(path, img)

        image = cv2.imread(img_path)

        if image is None:
            continue

        image = cv2.resize(image, (IMG_SIZE, IMG_SIZE))

        data.append(image)

        labels.append(idx)

X = np.array(data) / 255.0
y = np.array(labels)

from tensorflow.keras.utils import to_categorical

y = to_categorical(y, num_classes=3)

print("Dataset:", X.shape)

Dataset: (3000, 64, 64, 3)


In [93]:
X_train,X_test,y_train,y_test = train_test_split(
    X,y,test_size=0.2,random_state=42
)

In [94]:
# Data augmentation
datagen = ImageDataGenerator(

    rotation_range=20,
    zoom_range=0.2,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True

)

datagen.fit(X_train)

In [95]:

# CNN architecture
image_input = Input(shape=(64,64,3))

x = Conv2D(32,(3,3),activation='relu')(image_input)
x = BatchNormalization()(x)
x = MaxPool2D()(x)

x = Conv2D(64,(3,3),activation='relu')(x)
x = BatchNormalization()(x)
x = MaxPool2D()(x)

x = Conv2D(128,(3,3),activation='relu')(x)
x = BatchNormalization()(x)
x = MaxPool2D()(x)

x = Flatten()(x)

x = Dense(128,activation='relu')(x)
x = Dropout(0.5)(x)

feature_layer = Dense(32,activation='relu',name="feature_layer")(x)

output = Dense(3, activation='softmax')(feature_layer)
model = Model(inputs=image_input,outputs=output)

In [96]:

model.compile(
optimizer='adam',
loss='categorical_crossentropy',
metrics=['accuracy']
)


In [100]:

history = model.fit(

    datagen.flow(X_train,y_train,batch_size=32),

    validation_data=(X_test,y_test),

    epochs=25

)

Epoch 1/25
75/75 ━━━━━━━━━━━━━━━━━━━━ 18s 232ms/step - accuracy: 0.9183 - loss: 0.2221 - val_accuracy: 0.6983 - val_loss: 1.0420
Epoch 2/25
75/75 ━━━━━━━━━━━━━━━━━━━━ 17s 226ms/step - accuracy: 0.9217 - loss: 0.2357 - val_accuracy: 0.6217 - val_loss: 1.6937
Epoch 3/25
75/75 ━━━━━━━━━━━━━━━━━━━━ 17s 227ms/step - accuracy: 0.9262 - loss: 0.2190 - val_accuracy: 0.7667 - val_loss: 0.6233
Epoch 4/25
75/75 ━━━━━━━━━━━━━━━━━━━━ 22s 285ms/step - accuracy: 0.9217 - loss: 0.2067 - val_accuracy: 0.5883 - val_loss: 2.1622
Epoch 5/25
75/75 ━━━━━━━━━━━━━━━━━━━━ 20s 245ms/step - accuracy: 0.9229 - loss: 0.2229 - val_accuracy: 0.7633 - val_loss: 0.7890
Epoch 6/25
75/75 ━━━━━━━━━━━━━━━━━━━━ 19s 246ms/step - accuracy: 0.9358 - loss: 0.1877 - val_accuracy: 0.4000 - val_loss: 2.3008
Epoch 7/25
75/75 ━━━━━━━━━━━━━━━━━━━━ 19s 253ms/step - accuracy: 0.9321 - loss: 0.1855 - val_accuracy: 0.8983 - val_loss: 0.2478
Epoch 8/25
75/75 ━━━━━━━━━━━━━━━━━━━━ 17s 226ms/step - accuracy: 0.9425 - loss: 0.1689 - val_accu

In [102]:
loss,accuracy = model.evaluate(X_test,y_test)

print("Accuracy:",accuracy)

19/19 ━━━━━━━━━━━━━━━━━━━━ 1s 43ms/step - accuracy: 0.9050 - loss: 0.2602
Accuracy: 0.9049999713897705


In [104]:
model.save("cnn_model.h5")

print("Model saved as cnn_model.h5")

Model saved as cnn_model.h5


In [ ]:
#feature extracter
%%writefile feature_extractor.py
import numpy as np
import cv2
import pickle
from tensorflow.keras.models import Model

IMG_SIZE = 64

# Load model from pkl
with open("cnn_model.pkl", "rb") as f:
    model = pickle.load(f)

# Create feature extraction model
feature_model = Model(
    inputs=model.input,
    outputs=model.get_layer("feature_layer").output
)

def extract_features(image):

    img = cv2.resize(image,(IMG_SIZE,IMG_SIZE))
    img = img/255.0
    img = img.reshape(1,64,64,3)

    features = feature_model.predict(img)[0]

    commercial = np.mean(features[:10])
    traffic = np.mean(features[10:20])
    powerline = np.mean(features[20:32])

    return commercial,traffic,powerline


def predict_location(image):

    img = cv2.resize(image,(IMG_SIZE,IMG_SIZE))
    img = img/255.0
    img = img.reshape(1,64,64,3)

    score = model.predict(img)[0][0]

    commercial,traffic,powerline = extract_features(image)

    if score > 0.5:
        label = "Suitable"
    else:
        label = "Not Suitable"

    return label,score,commercial,traffic,powerline

Writing feature_extractor.py


In [ ]:
!pip install streamlit -q


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 65.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 88.9 MB/s eta 0:00:00


In [ ]:
#streamlit code
%%writefile app.py
import streamlit as st
import cv2
import numpy as np
from PIL import Image
from feature_extractor import predict_location

st.set_page_config(page_title="EV Charging Hub Predictor")

st.title("⚡ EV Charging Hub Location Predictor")

uploaded_file = st.file_uploader(
"Upload Satellite Image",
type=["jpg","png","jpeg"]
)

if uploaded_file:

    image = Image.open(uploaded_file)

    st.image(image,use_column_width=True)

    image_np = np.array(image)

    if st.button("Analyze Location"):

        label,score,commercial,traffic,powerline = predict_location(image_np)

        st.subheader("Prediction")

        st.write("Suitability Score:",round(score,2))

        st.success(label)

        st.subheader("Extracted Infrastructure Parameters")

        st.write("Commercial Activity:",round(commercial,2))
        st.progress(int(commercial*100))

        st.write("Traffic Density:",round(traffic,2))
        st.progress(int(traffic*100))

        st.write("Powerline Proximity:",round(powerline,2))
        st.progress(int(powerline*100))

Writing app.py


In [ ]:
%%writefile app.py
import streamlit as st

st.title("EV Charging Hub Predictor")

st.write("Streamlit is working successfully!")

Writing app.py


In [ ]:
!streamlit run app.py & npx localtunnel --port 8501

⠙⠹

⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏Need to install the following packages:
localtunnel@2.0.2
Ok to proceed? (y) 
  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.125.194.72:8501

  Stopping...
^C
